In [3]:
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.models import load_model
import pickle

In [4]:
## load the ANN trained model , scalar pickle , onehot
model = load_model('model.h5')

## load the label and onehot pickle file.
with open('label_encoder_gender.pkl' , 'rb') as file:
    label_encoder_gender = pickle.load(file)

with open('onehot_encoder_geo.pkl' , 'rb') as file :
    onehot_encoder_geo = pickle.load(file)

## load the scalar pickle file.
with open('scalar.pkl' , 'rb') as file:
    scalar = pickle.load(file)

In [5]:
input_data = {
"CreditScore":619,
"Geography":"France",
"Gender":"Female",
"Age":42,
"Tenure":2,
"Balance":0.0,
"NumOfProducts":1,
"HasCrCard":1,
"IsActiveMember":1,
"EstimatedSalary":101348.88
}

In [6]:
input_df = pd.DataFrame([input_data])

In [7]:
# Ome Hot Encoding on "Geography"
geo_encoded = onehot_encoder_geo.transform(input_df[['Geography']])
geo_encoded_df = pd.DataFrame(geo_encoded , columns=onehot_encoder_geo.get_feature_names_out(['Geography']))
geo_encoded_df

,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0


In [8]:
# Label Encoding on "Gender"
input_df['Gender'] = label_encoder_gender.transform(input_df[['Gender']])

d:\AI\Projects\DL Projects\Customer Churn Prediction\venv\Lib\site-packages\sklearn\preprocessing\_label.py:139: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, dtype=self.classes_.dtype, warn=True)


In [9]:
input_df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,619,France,0,42,2,0.0,1,1,1,101348.88


In [10]:
input_df = input_df.drop('Geography', axis=1)

In [11]:
input_df = pd.concat([input_df.reset_index(drop=True),
                      geo_encoded_df.reset_index(drop=True)], axis=1)

In [12]:
input_df

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.0,1,1,1,101348.88,1.0,0.0,0.0


In [13]:
# Scalling the input data
input_scaled = scalar.transform(input_df)
input_scaled

array([[-0.33880827, -1.09499335,  0.29493847, -1.04241787, -1.21847056,
        -0.91668767,  0.64920267,  0.97481699,  0.01595384,  1.00150113,
        -0.57946723, -0.57638802]])

In [14]:
# Predict Churn
prediction = model.predict(input_scaled)
prediction

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 154ms/step


array([[0.72022957]], dtype=float32)

In [15]:
prediction_prob = prediction[0][0]

In [16]:
prediction[0][0]

np.float32(0.72022957)

In [17]:
if prediction_prob > 0.5:
    print("Customer is likely to Churn.")
else: 
    print('Customer is not likely to Churn')

Customer is likely to Churn.
